# Kaggle GPU → Ollama API at a **fixed** address (Cloudflare or ngrok)

This notebook turns a free Kaggle GPU session (2× T4) into an Ollama server for **Qwen3.8 27B (Q4_K_M) + MTP**, reachable at the **same address every session**, protected by a token.

**Every session:** open the notebook → **Run All**. The address, the token and your client setup stay the same.

```
OmniRoute / any OpenAI client
   │  https://YOUR-FIXED-ADDRESS/v1   +  Bearer token
   ▼
Cloudflare named tunnel  OR  ngrok free domain   (set up once in the browser)
   ▼
Token check (this notebook, port 8000)  ── wrong/missing token → 401
   ▼
Ollama (this notebook, port 11434)  →  model on both T4 GPUs
```

Works with **anyone's** account: nothing personal is written in this notebook. Each person's details live in their own **Kaggle Secrets**.

## One-time setup

**What happens where**

| Step | Where | How often |
| --- | --- | --- |
| Create a tunnel account and copy its token/address | **Browser** (Cloudflare or ngrok website) | Once |
| Add Kaggle Secrets, turn on GPU + Internet | **Browser** (Kaggle) | Once |
| Install, log in, start tunnel, load model, check token | **This notebook** (Run All) | Every session |
| Point OmniRoute at the fixed address | **Browser** (OmniRoute) | Once |

### Step 1: pick ONE tunnel option

| | Option A: Cloudflare | Option B: ngrok |
| --- | --- | --- |
| Need | Cloudflare account **with your own domain** | Free ngrok account, **no domain needed** |
| Address | Your choice, e.g. `ollama.your-domain.com` | Assigned by ngrok, e.g. `abc123.ngrok-free.app` |
| Free limits | No bandwidth limit | About 1 GB outbound and 20,000 requests per month |
| Used when | `CF_TUNNEL_TOKEN` secret is present | `NGROK_AUTHTOKEN` secret is present (and no Cloudflare token) |

#### Option A: Cloudflare (browser, once)
Dashboard labels change from time to time; the steps stay the same.
1. **Cloudflare Zero Trust** dashboard → **Networks → Tunnels → Create a tunnel** → type **Cloudflared** → name it `kaggle-ollama` → save.
2. On the "install connector" screen, **don't install anything**. Copy the long **token** (the text after `--token`) → this is `CF_TUNNEL_TOKEN`.
3. Add a **Public Hostname**: subdomain `ollama`, your domain, service **HTTP**, URL **`localhost:8000`** → save.
4. Your address is `ollama.your-domain.com` → this is `PUBLIC_HOSTNAME`.

#### Option B: ngrok (browser, once)
1. Sign up free at **ngrok.com** (email or Google/GitHub login).
2. Dashboard → **Your Authtoken** → copy it → this is `NGROK_AUTHTOKEN`.
3. Dashboard → **Domains** → copy your free assigned domain, e.g. `abc123.ngrok-free.app` (claim it if asked) → this is `NGROK_DOMAIN`.
4. That's all. You do **not** install ngrok on your laptop; the notebook does it on Kaggle.

### Step 2: make an access token for the API (both options)
Any long random string. On Windows PowerShell:
```
[guid]::NewGuid().ToString('N') + [guid]::NewGuid().ToString('N')
```
This is `OLLAMA_API_TOKEN`. Anyone without it gets a 401.

### Step 3: add Kaggle Secrets (browser, once)
In this notebook: **Add-ons → Secrets → Add secret**, then make sure each one is **ticked (attached)** for this notebook.

| Secret | Option A: Cloudflare | Option B: ngrok |
| --- | --- | --- |
| `OLLAMA_API_TOKEN` | ✅ | ✅ |
| `CF_TUNNEL_TOKEN` | ✅ | – |
| `PUBLIC_HOSTNAME` | ✅ `ollama.your-domain.com` | – |
| `NGROK_AUTHTOKEN` | – | ✅ |
| `NGROK_DOMAIN` | – | ✅ `abc123.ngrok-free.app` |

Write domains without `https://`.

### Step 4: notebook settings (once)
**Settings → Accelerator → GPU T4 ×2**, and **Internet → On**. Kaggle needs a phone-verified account for both.

### Step 5: connect your client (browser, once)
In OmniRoute: **Providers → + Add OpenAI Compatible**
- Name: `kaggle` · Base URL: `https://YOUR-FIXED-ADDRESS/v1` · API key: your `OLLAMA_API_TOKEN`
- While this notebook is running: **Test → Import from /models**.

## Two ways to run it

| Mode | How | Runs while | Good for |
| --- | --- | --- | --- |
| **Interactive** | Open the notebook → **Run All** | The tab is open and the session is active | Quick use, testing |
| **Background** | **Save Version → Save & Run All (Commit)** | Up to Kaggle's session limit, no browser needed | Long sessions |

The last cell keeps the server running for `SERVE_HOURS`. In interactive mode, stop it any time with the **■ Stop** button. GPU time comes out of your weekly Kaggle quota, so stop sessions you are not using.

## Skip the 30-minute download: save the model once

Kaggle wipes the machine every new session, so the ~17 GB model would be downloaded from Hugging Face every time. Instead, save it once **inside Kaggle** and attach it to later sessions; each session then copies it from Kaggle's own storage, which takes a few minutes instead of about 30.

**Once (about 30–40 minutes, runs in the background):**
1. In the **Settings** cell below, set `BUILD_CACHE = True`.
2. Top right: **Save Version → Save & Run All (Commit)**. GPU and Internet stay on.
3. Wait until the version finishes ("Successfully ran"). This run only downloads and prepares the model; it does not start the tunnel.
4. Set `BUILD_CACHE = False` again.
5. Right panel: **Add Input → Your Work → Notebooks** → pick **this notebook's** saved version → **Add**.

**Every session after that:** Run All. The Settings cell finds the saved copy under `/kaggle/input/...` and links it in, so step 5 skips the download.

> The saved copy lives in the notebook's output, which Kaggle limits to about 20 GB. The model is about 17 GB, so it fits. If you change `SOURCE_MODEL`, build the cache again.

## 1. Settings

In [ ]:
# ---- Change these if you like -------------------------------------------------
SOURCE_MODEL  = "hf.co/JonathanColetti/Qwen3.8-27B-Uncensored-GGUF:Q4_K_M"
MTP_MODEL     = "qwen3.8-27b-uncensored-mtp"   # the name clients will use
NUM_CTX       = 65536                          # 64K context
DRAFT_TOKENS  = 2                              # MTP speculative tokens
SERVE_HOURS   = 11.5                           # how long the last cell keeps serving
BUILD_CACHE   = False                          # True = one-time run that saves the model for next sessions
PROXY_PORT    = 8000                           # must match the tunnel's Public Hostname service
# --------------------------------------------------------------------------------

import os, shutil

def read_secret(name, required=True):
    try:
        from kaggle_secrets import UserSecretsClient
        value = UserSecretsClient().get_secret(name)
        if value:
            return value.strip()
    except Exception:
        pass
    if required:
        print(f"⚠️  Secret '{name}' is missing or not attached (Add-ons → Secrets).")
    return None

CF_TUNNEL_TOKEN  = read_secret("CF_TUNNEL_TOKEN", required=False)
NGROK_AUTHTOKEN  = read_secret("NGROK_AUTHTOKEN", required=False)
OLLAMA_API_TOKEN = read_secret("OLLAMA_API_TOKEN")

clean = lambda v: (v or "").replace("https://", "").replace("http://", "").strip().strip("/")
if CF_TUNNEL_TOKEN:
    TUNNEL_MODE = "cloudflare"
    PUBLIC_HOSTNAME = clean(read_secret("PUBLIC_HOSTNAME"))
elif NGROK_AUTHTOKEN:
    TUNNEL_MODE = "ngrok"
    PUBLIC_HOSTNAME = clean(read_secret("NGROK_DOMAIN"))
else:
    TUNNEL_MODE = None
    PUBLIC_HOSTNAME = ""

import glob

# Model storage
#   BUILD_CACHE = True  -> /kaggle/working/ollama-models (saved as notebook output = the cache)
#   normal session      -> /tmp/ollama-models (large temporary disk, not saved)
if BUILD_CACHE:
    MODEL_DIR = "/kaggle/working/ollama-models"
else:
    MODEL_DIR = "/tmp/ollama-models"
    if shutil.disk_usage("/tmp").free / 1e9 < 25:
        MODEL_DIR = "/kaggle/working/ollama-models"
os.makedirs(MODEL_DIR, exist_ok=True)

# Saved copy from an earlier BUILD_CACHE run, attached via Add Input
CACHE_DIR = None
for pattern in ("/kaggle/input/*/ollama-models", "/kaggle/input/*/*/ollama-models", "/kaggle/input/*/*/*/ollama-models"):
    for path in glob.glob(pattern):
        if os.path.isdir(f"{path}/manifests") and os.path.isdir(f"{path}/blobs"):
            CACHE_DIR = path
            break
    if CACHE_DIR:
        break

if CACHE_DIR and not BUILD_CACHE:
    # Copy the saved model into the writable working folder.
    # (Kaggle inputs are read-only; Ollama must be able to update its files, so links do not work.)
    import time
    os.makedirs(f"{MODEL_DIR}/blobs", exist_ok=True)
    started = time.time()
    for name in os.listdir(f"{CACHE_DIR}/blobs"):
        source, target = f"{CACHE_DIR}/blobs/{name}", f"{MODEL_DIR}/blobs/{name}"
        if os.path.islink(target):
            os.remove(target)                      # left over from an older notebook version
        if os.path.exists(target) and os.path.getsize(target) == os.path.getsize(source):
            continue                               # already copied in this session
        size_gb = os.path.getsize(source) / 1e9
        if size_gb > 0.5:
            print(f"Copying saved model file ({size_gb:.1f} GB) ...", flush=True)
        shutil.copyfile(source, target)
        os.chmod(target, 0o644)
    shutil.copytree(f"{CACHE_DIR}/manifests", f"{MODEL_DIR}/manifests", dirs_exist_ok=True)
    print(f"Saved model copied in {time.time() - started:.0f}s")

OLLAMA_URL = "http://127.0.0.1:11434"
PUBLIC_URL = f"https://{PUBLIC_HOSTNAME}" if PUBLIC_HOSTNAME else None

print("Mode           :", "BUILD CACHE (one-time, saves the model)" if BUILD_CACHE else "serve")
print("Saved model    :", CACHE_DIR or "none attached - the model will be downloaded (~30 min)")
print("Model dir      :", MODEL_DIR, f"({shutil.disk_usage(MODEL_DIR).free/1e9:.0f} GB free)")
print("Tunnel         :", TUNNEL_MODE or "NONE - add Cloudflare or ngrok secrets for public access")
print("API token      :", "set" if OLLAMA_API_TOKEN else "MISSING - the public API would be OPEN")
print("Public address :", PUBLIC_URL or "(PUBLIC_HOSTNAME not set)")

## 2. Check the GPUs (expect two Tesla T4)

In [ ]:
import subprocess
gpus = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                      capture_output=True, text=True)
print(gpus.stdout or gpus.stderr)
if "T4" not in gpus.stdout:
    print("⚠️  No T4 found. Settings → Accelerator → GPU T4 ×2, then restart the session.")

## 3. Install Ollama (skipped if already installed)

In [ ]:
import shutil, subprocess
if not shutil.which("zstd"):
    subprocess.run("apt-get update -qq && apt-get install -y -qq zstd", shell=True, check=True)
if not shutil.which("ollama"):
    subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True)
OLLAMA_BIN = shutil.which("ollama")
print("Ollama:", OLLAMA_BIN, subprocess.run([OLLAMA_BIN, "--version"], capture_output=True, text=True).stdout.strip())

## 4. Start Ollama

In [ ]:
import os, subprocess, time, urllib.request, json

OLLAMA_ENV = os.environ.copy()
OLLAMA_ENV.update({
    "OLLAMA_HOST": "127.0.0.1:11434",
    "OLLAMA_MODELS": MODEL_DIR,
    "OLLAMA_KEEP_ALIVE": "-1",
    "OLLAMA_FLASH_ATTENTION": "1",
    "OLLAMA_KV_CACHE_TYPE": "q8_0",
    "OLLAMA_CONTEXT_LENGTH": str(NUM_CTX),
    "OLLAMA_NUM_PARALLEL": "1",
    "OLLAMA_MAX_LOADED_MODELS": "1",
    "OLLAMA_NOPRUNE": "1",      # never delete model files on start-up
})

def ollama_ready():
    try:
        urllib.request.urlopen(f"{OLLAMA_URL}/api/tags", timeout=2).close()
        return True
    except Exception:
        return False

def installed_models():
    with urllib.request.urlopen(f"{OLLAMA_URL}/api/tags", timeout=10) as r:
        return [m["name"] for m in json.loads(r.read()).get("models", [])]

def start_ollama():
    """Start 'ollama serve' if it is not answering; used again by step 10 if it ever dies."""
    global ollama_process
    if ollama_ready():
        return True
    ollama_log = open("/tmp/ollama-server.log", "a")
    ollama_process = subprocess.Popen([OLLAMA_BIN, "serve"], env=OLLAMA_ENV,
                                      stdout=ollama_log, stderr=subprocess.STDOUT)
    for _ in range(60):
        if ollama_ready():
            return True
        time.sleep(1)
    return False

if not start_ollama():
    raise RuntimeError("Ollama did not start. See /tmp/ollama-server.log")
print("✅ Ollama ready at", OLLAMA_URL)

## 5. Download the model (skipped if already present; ~17 GB)

In [ ]:
import subprocess
if any(n.startswith(SOURCE_MODEL) for n in installed_models()):
    print("✅ Model already downloaded")
else:
    if subprocess.run([OLLAMA_BIN, "pull", SOURCE_MODEL], env=OLLAMA_ENV).returncode != 0:
        # Workaround for some Ollama builds that hit a Hugging Face redirect error
        print("Standard pull failed; retrying with --insecure ...")
        subprocess.run([OLLAMA_BIN, "pull", "--insecure", SOURCE_MODEL], env=OLLAMA_ENV, check=True)
    print("✅ Model downloaded")

## 6. Create the MTP model and load it onto the GPUs

If MTP fails to create or load (it can crash on some GPU/driver combinations), this cell **automatically falls back to the base model**, restarts Ollama if needed, and tells you which model name clients should use.

In [ ]:
if BUILD_CACHE:
    print('⏭️  Cache build run: skipped. Set BUILD_CACHE = False for normal sessions.')
else:
    import json, urllib.request, urllib.error, time, subprocess

    def post(path, payload, timeout=900):
        req = urllib.request.Request(f"{OLLAMA_URL}{path}", data=json.dumps(payload).encode(),
                                     headers={"Content-Type": "application/json"}, method="POST")
        with urllib.request.urlopen(req, timeout=timeout) as r:
            return json.loads(r.read())

    def explain(e):
        try:
            return e.read().decode()[:300]
        except Exception:
            return str(e)

    def load(model, options):
        start = time.time()
        r = post("/api/generate", {"model": model, "prompt": "Reply with exactly: ready.", "stream": False,
                                    "keep_alive": -1, "options": options})
        print(f"✅ {model} loaded in {time.time()-start:.0f}s. Reply:", r.get("response", "").strip())

    SERVE_MODEL = SOURCE_MODEL          # what clients will call; switched to MTP_MODEL when MTP works
    mtp_ok = False
    try:
        if not any(n.split(":")[0] == MTP_MODEL for n in installed_models()):
            post("/api/create", {"model": MTP_MODEL, "from": SOURCE_MODEL, "stream": False,
                                 "parameters": {"draft_num_predict": DRAFT_TOKENS, "num_ctx": NUM_CTX}})
            print("✅ MTP model created:", MTP_MODEL)
        mtp_ok = True
    except Exception as e:
        print("⚠️  Could not create the MTP model:", explain(e))

    if BUILD_CACHE:
        print("✅ Cache build: model files are ready in", MODEL_DIR)
        print("   After this version finishes: set BUILD_CACHE = False and Add Input → this notebook's output.")
    else:
        if mtp_ok:
            try:
                load(MTP_MODEL, {"draft_num_predict": DRAFT_TOKENS, "num_ctx": NUM_CTX})
                SERVE_MODEL = MTP_MODEL
            except Exception as e:
                print("⚠️  MTP model failed to load:", explain(e))
                print("   Falling back to the base model (no MTP speed-up).")
        if SERVE_MODEL != MTP_MODEL:
            if not ollama_ready():
                print("   Ollama stopped during the MTP attempt - restarting it ...")
                start_ollama()
            load(SOURCE_MODEL, {"num_ctx": NUM_CTX})
        print("\nClients should use the model name:", SERVE_MODEL)
        print(subprocess.run([OLLAMA_BIN, "ps"], env=OLLAMA_ENV, capture_output=True, text=True).stdout)

## 7. Start the token check (port 8000)

Every request through the tunnel must carry `Authorization: Bearer <OLLAMA_API_TOKEN>`. Requests without it get **401** and never reach Ollama. Streaming replies pass straight through. `GET /health` is left open so you can check the server is up.

In [ ]:
if BUILD_CACHE:
    print('⏭️  Cache build run: skipped. Set BUILD_CACHE = False for normal sessions.')
else:
    import asyncio, threading, time, hmac, subprocess, sys

    try:
        from aiohttp import web, ClientSession, ClientTimeout
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "aiohttp"], check=True)
        from aiohttp import web, ClientSession, ClientTimeout

    HOP_HEADERS = {"host", "authorization", "content-length", "transfer-encoding", "connection", "accept-encoding"}

    async def handle(request):
        if request.path == "/health":
            return web.json_response({"status": "ok"})
        if OLLAMA_API_TOKEN:
            sent = request.headers.get("Authorization", "")
            if not hmac.compare_digest(sent, f"Bearer {OLLAMA_API_TOKEN}"):
                return web.json_response({"error": {"message": "invalid or missing API token"}}, status=401)
        headers = {k: v for k, v in request.headers.items() if k.lower() not in HOP_HEADERS}
        body = await request.read()
        try:
            async with request.app["session"].request(
                    request.method, f"{OLLAMA_URL}{request.rel_url.path_qs}", headers=headers, data=body) as up:
                resp = web.StreamResponse(status=up.status)
                if "Content-Type" in up.headers:
                    resp.headers["Content-Type"] = up.headers["Content-Type"]
                await resp.prepare(request)
                async for chunk in up.content.iter_any():
                    await resp.write(chunk)
                await resp.write_eof()
                return resp
        except Exception as e:
            return web.json_response({"error": {"message": f"Ollama is not answering inside Kaggle: {type(e).__name__}"}},
                                     status=502)

    def run_proxy():
        loop = asyncio.new_event_loop()
        asyncio.set_event_loop(loop)
        app = web.Application(client_max_size=64 * 1024 * 1024)
        async def on_startup(app): app["session"] = ClientSession(timeout=ClientTimeout(total=None))
        async def on_cleanup(app): await app["session"].close()
        app.on_startup.append(on_startup)
        app.on_cleanup.append(on_cleanup)
        app.router.add_route("*", "/{tail:.*}", handle)
        runner = web.AppRunner(app)
        loop.run_until_complete(runner.setup())
        loop.run_until_complete(web.TCPSite(runner, "127.0.0.1", PROXY_PORT).start())
        loop.run_forever()

    def proxy_up():
        try:
            urllib.request.urlopen(f"http://127.0.0.1:{PROXY_PORT}/health", timeout=2).close()
            return True
        except Exception:
            return False

    if not proxy_up():
        threading.Thread(target=run_proxy, daemon=True).start()
        for _ in range(20):
            if proxy_up():
                break
            time.sleep(0.5)
    print("✅ Token check running on port", PROXY_PORT if proxy_up() else "- FAILED")
    if not OLLAMA_API_TOKEN:
        print("⚠️  OLLAMA_API_TOKEN not set: anyone with the address can use this server.")

## 8. Start the tunnel (Cloudflare or ngrok, fixed address)

In [ ]:
if BUILD_CACHE:
    print('⏭️  Cache build run: skipped. Set BUILD_CACHE = False for normal sessions.')
else:
    import subprocess, shutil, time, re, json, urllib.request

    def install_cloudflared():
        if not shutil.which("cloudflared"):
            subprocess.run("curl -fsSL -o /tmp/cloudflared.deb "
                           "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb"
                           " && dpkg -i /tmp/cloudflared.deb", shell=True, check=True)

    def install_ngrok():
        if not shutil.which("ngrok"):
            subprocess.run("curl -fsSL -o /tmp/ngrok.tgz "
                           "https://bin.equinox.io/c/bNyj1mQVY4c/ngrok-v3-stable-linux-amd64.tgz"
                           " && tar -xzf /tmp/ngrok.tgz -C /usr/local/bin", shell=True, check=True)
        subprocess.run(["ngrok", "config", "add-authtoken", NGROK_AUTHTOKEN],
                       check=True, capture_output=True)

    def tunnel_running():
        return "tunnel_process" in globals() and tunnel_process.poll() is None

    def ngrok_public_url():
        try:
            with urllib.request.urlopen("http://127.0.0.1:4040/api/tunnels", timeout=2) as r:
                tunnels = json.loads(r.read()).get("tunnels", [])
            return next((t["public_url"] for t in tunnels if t["public_url"].startswith("https://")), None)
        except Exception:
            return None

    def start_tunnel():
        global tunnel_process
        log_path = "/tmp/tunnel.log"
        if TUNNEL_MODE == "cloudflare":
            commands = [["cloudflared", "tunnel", "--no-autoupdate", "--protocol", "http2",
                         "run", "--token", CF_TUNNEL_TOKEN]]
        else:
            # Newer ngrok uses --url, older builds use --domain: try both
            commands = [["ngrok", "http", str(PROXY_PORT), "--url", f"https://{PUBLIC_HOSTNAME}", "--log", "stdout"],
                        ["ngrok", "http", str(PROXY_PORT), "--domain", PUBLIC_HOSTNAME, "--log", "stdout"]]
        for cmd in commands:
            log = open(log_path, "w")
            tunnel_process = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT)
            for _ in range(60):
                time.sleep(1)
                if TUNNEL_MODE == "cloudflare":
                    if "Registered tunnel connection" in open(log_path).read():
                        return True
                elif ngrok_public_url():
                    return True
                if tunnel_process.poll() is not None:
                    break
            if tunnel_process.poll() is None:
                tunnel_process.terminate()
        print(open(log_path).read()[-1500:])
        return False

    if not TUNNEL_MODE:
        print("⏭️  No tunnel secrets: the server works inside Kaggle only.")
    elif not PUBLIC_HOSTNAME:
        print("❌ Missing", "PUBLIC_HOSTNAME" if TUNNEL_MODE == "cloudflare" else "NGROK_DOMAIN", "secret.")
    elif tunnel_running():
        print(f"✅ {TUNNEL_MODE} tunnel already running")
    else:
        install_cloudflared() if TUNNEL_MODE == "cloudflare" else install_ngrok()
        print(f"✅ {TUNNEL_MODE} tunnel connected" if start_tunnel()
              else f"❌ {TUNNEL_MODE} tunnel did not connect - check the token and domain secrets")

    if PUBLIC_URL:
        print("\nBase URL for OmniRoute / OpenAI clients:", f"{PUBLIC_URL}/v1")
        print("Model name                            :", SERVE_MODEL)
        print("API key                               : your OLLAMA_API_TOKEN secret")

## 9. Test from outside (through Cloudflare)

In [ ]:
if BUILD_CACHE:
    print('⏭️  Cache build run: skipped. Set BUILD_CACHE = False for normal sessions.')
else:
    import json, urllib.request, urllib.error

    def public_call(path, payload=None, token=OLLAMA_API_TOKEN):
        req = urllib.request.Request(f"{PUBLIC_URL}{path}",
                                     data=json.dumps(payload).encode() if payload else None,
                                     headers={"Content-Type": "application/json", "ngrok-skip-browser-warning": "true",
                                              **({"Authorization": f"Bearer {token}"} if token else {})},
                                     method="POST" if payload else "GET")
        with urllib.request.urlopen(req, timeout=300) as r:
            return json.loads(r.read())

    if not PUBLIC_URL:
        print("Set the PUBLIC_HOSTNAME secret to test the public address.")
    else:
        # 1. Without a token: should be refused
        try:
            public_call("/v1/models", token=None)
            print("❌ Request WITHOUT token was accepted - check OLLAMA_API_TOKEN")
        except urllib.error.HTTPError as e:
            print("✅ Without token:", e.code, "(refused, as intended)")
        # 2. With the token: should work, OpenAI-style
        models = [m["id"] for m in public_call("/v1/models")["data"]]
        print("✅ With token, models:", models)
        r = public_call("/v1/chat/completions", {"model": SERVE_MODEL, "max_tokens": 40,
                        "messages": [{"role": "user", "content": "Say hello in five words."}]})
        print("✅ Chat reply:", r["choices"][0]["message"]["content"].strip())

## 10. Keep serving

Keeps the session alive for `SERVE_HOURS`, checks Ollama and the tunnel every 5 minutes, and restarts the tunnel or reloads the model if needed.

- **Interactive:** press **■ Stop** when you are done.
- **Commit (background):** this cell is what keeps the session running.

In [ ]:
if BUILD_CACHE:
    print('⏭️  Cache build run: skipped. Set BUILD_CACHE = False for normal sessions.')
else:
    import time, datetime

    end = time.time() + SERVE_HOURS * 3600
    print(f"Serving until about {datetime.datetime.now() + datetime.timedelta(hours=SERVE_HOURS):%H:%M} (Kaggle time).")
    try:
        while time.time() < end:
            status = []
            if not ollama_ready():
                status.append("Ollama restarted" if start_ollama() else "Ollama DOWN - see /tmp/ollama-server.log")
            if ollama_ready():
                try:
                    post("/api/generate", {"model": SERVE_MODEL, "prompt": "", "keep_alive": -1}, timeout=600)
                    status.append(f"{SERVE_MODEL.split('/')[-1]} loaded")
                except Exception as e:
                    status.append(f"reload failed: {e}")
            if not proxy_up():
                threading.Thread(target=run_proxy, daemon=True).start()
                status.append("token check restarted")
            if TUNNEL_MODE and PUBLIC_HOSTNAME and not tunnel_running():
                status.append("tunnel restarted" if start_tunnel() else "tunnel DOWN")
            print(f"{datetime.datetime.now():%H:%M}  " + (", ".join(status) or "ok"), flush=True)
            time.sleep(300)
    except KeyboardInterrupt:
        print("Stopped.")